## Dataset Description

The Leuven.cool dataset contains meteorological measurements collected
from a network of urban sensors.

Below are the key variables used in this project:

| Column Name | Description | Unit | Usage |
|------------|------------|------|-------|
| ID | Unique identifier of each sensor station | - | Used to group measurements and merge with station metadata |
| TEMPF | Temperature measured at the sensor | Fahrenheit (°F) | Converted to Celsius and used as target variable |
| DATEUTC | Timestamp of measurement in UTC | datetime | Used to filter data for a specific date |
| LATITUDE | Geographic latitude of the station | degrees | Used for spatial mapping |
| LONGITUDE | Geographic longitude of the station | degrees | Used for spatial mapping |
| temp_c | Temperature converted to Celsius | Celsius (°C) | Final target variable for the model |

### Notes

- Temperature is originally recorded in Fahrenheit and converted to Celsius
  for easier interpretation.
- The dataset contains high-frequency measurements, which are aggregated
  to obtain a single value per sensor.
- Geographic coordinates are obtained from a separate metadata file
  and merged using the station ID.

In [34]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pyproj import Transformer

### Loading Satellite Data (Sentinel-2)

In this step, we load satellite imagery from the Copernicus Sentinel-2 dataset.

We specifically use:
- **Band 4 (Red)**  
- **Band 8 (Near Infrared)**  

These bands are essential for computing vegetation indices such as NDVI, which will later help us analyze the relationship between vegetation and temperature.

In [35]:
from pathlib import Path

BASE_DIR = Path().resolve().parent

B04_path = BASE_DIR / "data/raw/sentinel2/B04.jp2"
B08_path = BASE_DIR / "data/raw/sentinel2/B08.jp2"

with rasterio.open(B04_path) as src:
    red = src.read(1)

with rasterio.open(B08_path) as src:
    nir = src.read(1)

print("Shape:", red.shape)

Shape: (10980, 10980)


## NDVI Computation

To analyze vegetation coverage, we compute the Normalized Difference Vegetation Index (NDVI).

NDVI is calculated using the Red (B04) and Near-Infrared (B08) bands:

- Vegetation reflects more NIR and absorbs red light
- This makes NDVI a good indicator of plant health

Expected values:
- Close to +1 → dense vegetation
- Around 0 → bare soil
- Negative → water or urban surfaces

In [36]:
red = red.astype(np.float32)
nir = nir.astype(np.float32)

ndvi = (nir - red) / (nir + red + 1e-6)

## Loading Leuven.cool Dataset

We load temperature measurements from Leuven.cool, which provides
ground-based sensor data across the city.

This data will serve as ground truth for training our model.

In [37]:
df = pd.read_csv("../data/raw/leuven/RAWDATA2025Q3.csv", nrows=100000)
stations = pd.read_csv("../data/raw/leuven/stations.csv")

print(df.columns)
print(stations.columns)

Index(['OBSID', 'ID', 'TEMPF', 'HUMIDITY', 'DEWPTF', 'WINDCHILLF', 'WINDDIR',
       'WINDSPEEDMPH', 'WINDGUSTMPH', 'RAININ', 'DAILYRAININ',
       'SOLARRADIATION', 'UV', 'BAROMIN', 'DATEUTC', 'DATEUTC_STAT'],
      dtype='str')
Index(['ID', 'WOWID', 'LATITUDE', 'LONGITUDE', 'ALTITUDE'], dtype='str')


## Filtering Temperature Data

To align the ground sensor measurements with the satellite image,
we filter the dataset for the closest available date.

Although the satellite image was acquired on 2025-07-02, the sensor
data is only available for 2025-07-01. Therefore, we use this date
as an approximation.

This is acceptable because temperature patterns change gradually,
and the selected date is temporally close to the satellite observation.

In [38]:
# Convert temperature to Celsius
df['temp_c'] = (df['TEMPF'] - 32) * 5/9

# Convert timestamp
df['DATEUTC'] = pd.to_datetime(df['DATEUTC'])

# Filter for closest available date
df_day = df[
    (df['DATEUTC'] >= '2025-07-01') &
    (df['DATEUTC'] < '2025-07-02')
]

print(df_day.head())

        OBSID         ID  TEMPF  HUMIDITY  DEWPTF  WINDCHILLF  WINDDIR  \
0  1057644192  GARMON062   74.8        69    64.0        74.8      117   
1  1057644193  GARMON043   70.5        82    64.8        70.5      242   
2  1057644194  GARMON045   68.9        83    63.5        68.9       30   
3  1057644195  GARMON114   77.9        63    64.4        77.9      306   
4  1057644196  GARMON057   72.3        75    64.0        72.3      230   

   WINDSPEEDMPH  WINDGUSTMPH  RAININ  DAILYRAININ  SOLARRADIATION  UV  \
0           0.0          0.0       0          0.0             0.0   0   
1           0.0          0.0       0          0.0             0.0   0   
2           0.0          0.0       0          0.0             0.0   0   
3           0.0          0.0       0          0.0             0.0   0   
4           0.0          0.0       0          0.0             0.0   0   

   BAROMIN             DATEUTC         DATEUTC_STAT     temp_c  
0      NaN 2025-07-01 00:00:00  2025-07-01 00:00:00

### Interpretation

The filtered dataset contains temperature measurements for July 1, 2025.

These measurements represent early morning conditions, which may differ
from peak daytime temperatures. However, they still provide valuable
information about spatial temperature variation across the city.

Next, we aggregate the data to obtain a single temperature value per sensor.

In [39]:
df_clean = df_day.groupby('ID')['temp_c'].mean().reset_index()

print(df_clean.head())

          ID     temp_c
0  GARMON002  18.129876
1  GARMON003  18.818496
2  GARMON004  20.017753
3  GARMON008  19.555674
4  GARMON009  19.915654


## Merging Temperature Data with Station Locations

We merge the temperature measurements with station metadata
to associate each sensor with its geographic coordinates.

In [40]:
df_merged = df_clean.merge(stations, on="ID")

# Rename for clarity
df_merged = df_merged.rename(columns={
    "LATITUDE": "lat",
    "LONGITUDE": "lon"
})

print(df_merged.head())

          ID     temp_c                                 WOWID      lat    lon  \
0  GARMON002  18.129876  83a89aa6-2695-e911-80e7-0003ff59883f  50.8468  4.756   
1  GARMON003  18.818496  2a3596b2-2795-e911-80e7-0003ff59889d  50.8700  4.728   
2  GARMON004  20.017753  7d43d8ab-2895-e911-80e7-0003ff59883f  50.8708  4.685   
3  GARMON008  19.555674  e9a45d8e-2995-e911-80e7-0003ff59883f  50.8748  4.663   
4  GARMON009  19.915654  3b54a9fd-5796-e911-80e7-0003ff59889d  50.8711  4.714   

   ALTITUDE  
0        47  
1        44  
2        31  
3        51  
4        38  


### Interpretation

The merged dataset now contains:

- Temperature values (target variable)
- Geographic coordinates (latitude and longitude)

This allows us to link ground measurements with satellite imagery,
which is essential for training our machine learning model.

---

## Coordinate Transformation

Satellite imagery uses a projected coordinate system (UTM),
while sensor data is in geographic coordinates (latitude, longitude).

We convert sensor coordinates to the same CRS as the satellite image
before mapping them to pixel locations.

In [41]:
with rasterio.open(B04_path) as src:
    transform = src.transform
    crs = src.crs

print(crs)

EPSG:32631


In [48]:
# Convert coordinates
# Transformer: WGS84 → Image CRS
transformer = Transformer.from_crs("EPSG:4326", crs, always_xy=True)

def latlon_to_pixel(lat, lon):
    x, y = transformer.transform(lon, lat)  # convert to UTM
    col, row = ~transform * (x, y)         # convert to pixel
    return int(row), int(col)


df_merged["row"], df_merged["col"] = zip(*df_merged.apply(
    lambda x: latlon_to_pixel(x["lat"], x["lon"]),
    axis=1
))

print(df_merged[['row', 'col']].head())

    row   col
0  6574  2361
1  6320  2158
2  6318  1856
3  6278  1700
4  6310  2059


### Interpretation

Each sensor location is now mapped to a pixel coordinate in the
satellite image.

- `row` corresponds to vertical position
- `col` corresponds to horizontal position

This allows us to extract image patches centered around each sensor.

In [50]:
print(df_merged[['row', 'col']].describe())
print("Image shape:", red.shape)
print(df_merged[['row','col']].head())

                row          col
count     75.000000    75.000000
mean    6338.360000  2064.186667
std      922.359074   899.714349
min     4893.000000  -436.000000
25%     6150.500000  1889.500000
50%     6273.000000  1982.000000
75%     6382.500000  2123.500000
max    13740.000000  7221.000000
Image shape: (10980, 10980)
    row   col
0  6574  2361
1  6320  2158
2  6318  1856
3  6278  1700
4  6310  2059


### Validation

We verify that the computed pixel coordinates fall within the image bounds.

Valid pixel indices should lie within the dimensions of the satellite image.

---

## Extracting Image Patches

To train a machine learning model, we extract small image patches
around each sensor location.

Each patch represents the local spatial environment (vegetation,
urban structures), which influences temperature.

These patches will serve as input features, while the corresponding
sensor temperature will serve as the target variable.

In [52]:
image_stack = np.stack([red, nir, ndvi], axis=0)

print("Image stack shape:", image_stack.shape)

Image stack shape: (3, 10980, 10980)


In [53]:
patch_size = 16

patches = []
temps = []

for _, row in df_merged.iterrows():
    r, c = row["row"], row["col"]
    
    patch = image_stack[:, 
                        r - patch_size//2 : r + patch_size//2,
                        c - patch_size//2 : c + patch_size//2]
    
    # Keep only valid patches
    if patch.shape == (3, patch_size, patch_size):
        patches.append(patch)
        temps.append(row["temp_c"])

patches = np.array(patches)
temps = np.array(temps)

print("Patches shape:", patches.shape)
print("Temps shape:", temps.shape)

Patches shape: (74, 3, 16, 16)
Temps shape: (74,)
